# Notebook 1: JAX Crash Course

This notebook gives a very short introduction to the JAX concepts that we will use later. The goal is not to learn all of JAX, but to understand how it differs from ordinary Python and NumPy.

The main idea is:

> In JAX, pure functions can be transformed.

We will see three central transformations:

- `jax.grad`
- `jax.jit`
- `jax.vmap`

together with JAX's explicit treatment of randomness.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

## 1. JAX looks like NumPy

JAX provides a NumPy-like interface through `jax.numpy`.

For many simple array operations, replacing

```python
import numpy as np
```

by

```python
import jax.numpy as jnp
```

is enough. However, JAX arrays are not NumPy arrays. JAX is designed for automatic differentiation, compilation, vectorisation, and accelerator hardware.

In [4]:
x_np = np.arange(5)
x_jax = jnp.arange(5)

print(x_np)
print(type(x_np))

print(x_jax)
print(type(x_jax))

[0 1 2 3 4]
<class 'numpy.ndarray'>
[0 1 2 3 4]
<class 'jaxlib._jax.ArrayImpl'>


In [5]:
x = jnp.arange(10)

print("x =", x)
print("mean =", jnp.mean(x))
print("sum =", jnp.sum(x))
print("norm =", jnp.linalg.norm(x))

x = [0 1 2 3 4 5 6 7 8 9]
mean = 4.5
sum = 45
norm = 16.881943


## 2. JAX arrays are immutable

One important difference from NumPy is that JAX arrays cannot be modified in place.

This design enables transformations such as automatic differentiation and compilation, while encouraging an array-oriented programming style that is well suited to parallel hardware such as GPUs.

In [7]:
x = jnp.arange(5)

try:
    x[0] = 42
except Exception as e:
    print(type(e).__name__)
    print(e)

TypeError
JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html


Instead of modifying an array in place, JAX returns a new array.

In [8]:
x = jnp.arange(5)

y = x.at[0].set(42)

print("Original array:", x)
print("Updated array:", y)

Original array: [0 1 2 3 4]
Updated array: [42  1  2  3  4]


## 3. Automatic differentiation with `jax.grad`

One of the main motivations for using JAX is automatic differentiation.

Given a Python function, `jax.grad` constructs another function that evaluates its derivative.

In [10]:
def f(x):
    return jnp.sin(x) + x**2

df = jax.grad(f)

x = 1.0

print("f(x)  =", f(x))
print("f'(x) =", df(x))

f(x)  = 1.841471
f'(x) = 2.5403023


The function passed to `jax.grad` must return a scalar. The input, however, can be an array.

In [11]:
def f(x):
    return jnp.sum(x**2)

x = jnp.array([1.0, 2.0, 3.0])

jax.grad(f)(x)

Array([2., 4., 6.], dtype=float32)

## 4. Just-in-time compilation with `jax.jit`

JAX can compile numerical functions using `jax.jit`. The important point is that `jax.jit` transforms a Python function into a compiled function.

In [12]:
def f(x):
    return jnp.sum(jnp.sin(x) ** 2)

f_jit = jax.jit(f)

x = jnp.linspace(0.0, 10.0, 1_000_000)

print(f(x))
print(f_jit(x))

477176.0
477176.0


The first call to a jitted function includes compilation. Subsequent calls reuse the compiled version.

In [13]:
%time f_jit(x).block_until_ready()
%time f_jit(x).block_until_ready()

CPU times: user 4.98 ms, sys: 1.67 ms, total: 6.64 ms
Wall time: 1.54 ms
CPU times: user 3.61 ms, sys: 64 μs, total: 3.68 ms
Wall time: 1 ms


Array(477176., dtype=float32)

## 5. Vectorisation with `jax.vmap`

Scientific computations often involve applying the same function to many inputs. A Python loop works, but it does not expose much parallelism to JAX. The transformation `jax.vmap` turns a function acting on one input into a function acting on a batch of inputs.

In [15]:
def f(x):
    return jnp.sin(x) + x**2

xs = jnp.linspace(-2.0, 2.0, 10)

f_batch = jax.vmap(f)

f_batch(xs)

Array([ 3.0907025 ,  1.4198692 ,  0.33837575, -0.1739254 , -0.17101505,
        0.26978055,  1.0628145 ,  2.1307602 ,  3.419637  ,  4.9092975 ],      dtype=float32)

Function transformations can be combined.

In [16]:
df = jax.grad(f)
df_batch = jax.vmap(df)
df_batch_jit = jax.jit(df_batch)

df_batch_jit(xs)

Array([-4.4161468 , -3.095871  , -1.7785563 , -0.54744583,  0.53096557,
        1.4198546 ,  2.1192207 ,  2.6658883 ,  3.1263514 ,  3.5838532 ],      dtype=float32)

## 6. Random numbers

Random number generation is another important difference between NumPy and JAX.

In NumPy, random numbers are usually generated from a hidden global state. In JAX, randomness is explicit: random numbers are generated from PRNG keys.

In [17]:
key = jax.random.PRNGKey(0)

key

Array([0, 0], dtype=uint32)

In [18]:
jax.random.normal(key)

Array(1.6226422, dtype=float32)

Keys should not be reused. Instead, we split keys into new independent keys.

In [19]:
key = jax.random.PRNGKey(0)

for _ in range(3):
    key, subkey = jax.random.split(key)
    print(jax.random.normal(subkey))

-2.4424558
-1.2574776
-1.3877681


Random arrays are generated in the same way.

In [20]:
key, subkey = jax.random.split(key)

jax.random.normal(subkey, shape=(5,))

Array([-2.3022664 ,  0.05277798,  1.6845714 , -0.4193235 , -0.07544231],      dtype=float32)

## Summary

In this notebook, we introduced some core ideas behind JAX. The main take-home message is that, in JAX, pure functions can be transformed, and these transformations can be freely combined to build efficient and differentiable numerical programs.

In the next notebook, we will use these tools to implement a minimal Variational Monte Carlo algorithm from scratch.